<!-- COMMONS LAUNCHER v3 · generated by tools/notebooks.py · do not edit by hand -->
<a href="https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials"><img src="https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/brand/synapsa-commons-badge.png" alt="Synapsa Commons" height="36"></a>

Free, hands-on AI courses that run anywhere, from the team building Synapsa, an AI-native
learning platform.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/programmes/model-risk/lessons/P04-L07-challenger-from-scratch/lesson.ipynb)
[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/blob/master/programmes/model-risk/lessons/P04-L07-challenger-from-scratch/lesson.ipynb)
[![Open in Binder](https://mybinder.org/badge_logo.svg)](https://mybinder.org/v2/gh/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master?labpath=programmes/model-risk/lessons/P04-L07-challenger-from-scratch/lesson.ipynb)
[![Open in Codespaces](https://github.com/codespaces/badge.svg)](https://codespaces.new/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials)

This lesson needs Python 3.11 or newer with numpy and matplotlib, which Colab, Kaggle,
Binder and Codespaces already have.

In [ ]:
# --- COMMONS LAUNCHER v3 · generated by tools/notebooks.py · do not edit by hand ---
# Makes this notebook run anywhere. Every line is a no-op when the thing is already present,
# so a local clone pays nothing and an online notebook repairs itself.
import importlib.util, os, subprocess, sys, urllib.request
from pathlib import Path

COMMONS_PIP = []            # (import name, pinned pip spec) for what this lesson imports
COMMONS_SIBLINGS = []    # files that must sit beside the notebook
# A fork, a classroom mirror or an offline copy can serve the files from elsewhere by setting
# COMMONS_RAW_OVERRIDE before running this cell.
COMMONS_RAW = os.environ.get("COMMONS_RAW_OVERRIDE") or "https://raw.githubusercontent.com/TarrySingh/Artificial-Intelligence-Deep-Learning-Machine-Learning-Tutorials/master/programmes/model-risk/lessons/P04-L07-challenger-from-scratch/"

# Resolve siblings against the LESSON's own directory, not the working directory. A notebook
# has no __file__ and runs with cwd alongside itself; a grader imports this file from the repo
# root. Checking cwd blindly makes the grader think every sibling is missing and reach for the
# network -- which would put a download on a graded path.
try:
    COMMONS_DIR = Path(__file__).resolve().parent
except NameError:
    COMMONS_DIR = Path.cwd()


def commons_host() -> str:
    """Name the notebook service we are on. Used for the message, and for honest errors."""
    try:
        if importlib.util.find_spec("google.colab") is not None:
            return "Google Colab"
    except (ImportError, ValueError):
        pass
    if os.environ.get("KAGGLE_KERNEL_RUN_TYPE"):
        return "Kaggle"
    if os.environ.get("BINDER_SERVICE_HOST"):
        return "Binder"
    if os.environ.get("CODESPACES"):
        return "GitHub Codespaces"
    return "a local Python environment"


_missing = [pip for imp, pip in COMMONS_PIP if importlib.util.find_spec(imp) is None]
if _missing:
    print("installing " + ", ".join(_missing) + " ...")
    # pip everywhere a student is likely to be; uv-managed local venvs ship without pip.
    if importlib.util.find_spec("pip") is not None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_missing], check=True)
    else:
        subprocess.run(["uv", "pip", "install", "-q", "--python", sys.executable, *_missing],
                       check=True)
    importlib.invalidate_caches()

_fetched = []
for _name in COMMONS_SIBLINGS:
    if not (COMMONS_DIR / _name).exists():
        (COMMONS_DIR / _name).parent.mkdir(parents=True, exist_ok=True)
        try:
            urllib.request.urlretrieve(COMMONS_RAW + _name, COMMONS_DIR / _name)
            _fetched.append(_name)
        except Exception as _e:  # Kaggle disables the internet by default; say so plainly
            raise RuntimeError(
                f"this lesson needs {_name} beside the notebook and could not fetch it "
                f"({_e}). On Kaggle, switch Internet on in the notebook settings panel "
                f"(Kaggle allows that only for phone-verified accounts); otherwise download it "
                f"from {COMMONS_RAW + _name} and upload it beside the notebook."
            ) from None

print("ready on " + commons_host() + ("; fetched " + ", ".join(_fetched) if _fetched else ""))
# --- END COMMONS LAUNCHER ---

# P04-L07 · The challenger, built from scratch

**You will build:** logistic regression by iteratively reweighted least squares, with a
convergence test you can state and a separation check that refuses to report a coefficient
that has no finite value; the two checks that prove a fit is the maximum it claims to be; a
monotonic weight-of-evidence scorecard as a second, more conservative challenger; a
comparability audit that answers the first question a committee asks; and the comparison
itself, run with module 4's paired bootstrap and module 1's promotion rule.

**Time:** ~90 minutes · **Runs on:** a laptop CPU, 8 GiB RAM, no GPU, no download
· **Prerequisites:** T00-L01 (the tier gate and the profiler), P04-L01 (the validation
suite, whose promotion rule you carry here) and P04-L04 (discrimination testing, whose
paired bootstrap you carry here). Pure numpy: no scikit-learn, no statsmodels, no scipy.

Module 1 handed you a challenger's scores. A benchmark model you did not build is not a
benchmark; it is another vendor claim. This lesson builds two, and then asks whether they
were given a fair fight. The data is **synthetic and generated in this notebook**; every
figure you see is computed by code you run.

By the end you will be able to:

1. Implement logistic regression by IRLS, solving the weighted normal equations without an
   explicit inverse and stopping on a stated criterion.
2. Detect complete and quasi-complete separation and report it as a finding, not a number.
3. Verify a fit by its score equations and by the closed form of a 2×2 table.
4. Implement a binned weight-of-evidence scorecard whose monotonicity is enforced by rule.
5. Audit a comparison for the same records, period and missing values — and explain why a
   comparison that fails the audit must report no metrics and no promotion decision.

In [ ]:
# Setup: everything the lesson needs, in one cell, with versions printed.
import contextlib
import io
import math
import sys
import time
import traceback
from typing import Callable, Mapping, NamedTuple, Sequence

import numpy as np

SEED = 20260923
N_DEV = 4000            # development book: the year the challengers are built on
N_VAL = 3000            # validation book: the later half-year every model is judged on
DEV_MONTHS = tuple(f"2024-{m:02d}" for m in range(1, 13))
VAL_MONTHS = tuple(f"2025-{m:02d}" for m in range(1, 7))

TOL = 1e-8              # IRLS stops when no coefficient moves by more than this in one update
MAX_ITER = 25           # ... or after this many updates, whichever comes first
SEPARATION_EPS = 1e-6   # a fitted probability of a record's own outcome this close to 1 is "perfect"

N_WOE_BINS = 5          # value bins per scorecard variable, before monotonic pooling
WOE_SMOOTHING = 0.5     # added to both class counts of every occupied bin

N_BINS = 10             # module 1's reliability bins, for the ECE
N_BOOT = 1000           # paired bootstrap replicates
ALPHA = 0.05            # two-sided: the paired interval runs from ALPHA/2 to 1 - ALPHA/2
POLICY = {"min_auc": 0.65, "max_ece": 0.04, "min_auc_gain": 0.02}   # module 1's rule

INPUTS = ("utilisation", "income", "delinquencies", "months_on_book")

_LESSON_T0 = time.perf_counter()
print("python", sys.version.split()[0], "· numpy", np.__version__)

DATA_NOTE = (
    "SYNTHETIC DATA. Every record in this notebook was generated inside it by "
    f"numpy.random.default_rng({SEED}). No real applicant, account, bureau file or vendor "
    "score is represented. The champion's coefficients are this notebook's own invention, "
    "written down as if they were the model inventory's documentation."
)

_FAILED_CHECKS: list[str] = []

# The exercises, in the order you meet them, and the functions each one asks you to write.
# The progress board at the foot of the notebook is built from this, and a cell that is
# waiting on an unfinished exercise names it from here.
_EXERCISES: dict[str, tuple[str, ...]] = {
    "exercise 1": ("design_matrix",),
    "exercise 2": ("fit_logistic_irls",),
    "exercise 3": ("score_equations", "closed_form_2x2"),
    "exercise 4": ("woe_from_counts", "woe_table"),
    "exercise 5": ("enforce_monotonic",),
    "exercise 6": ("comparability_audit",),
    "exercise 7": ("challenger_comparison",),
}
_STATUS: dict[str, str] = {}   # label -> "passed" | "failed" | "not started", latest run


def _named(labels: list[str]) -> str:
    """["exercise 3"] -> "exercise 3 (score_equations, closed_form_2x2)"; several -> "exercises 2 and 5"."""
    if len(labels) == 1:
        return f"{labels[0]} ({', '.join(_EXERCISES[labels[0]])})"
    nums = [label.split()[-1] for label in labels]
    return "exercises " + ", ".join(nums[:-1]) + " and " + nums[-1]


def _try(label: str, check: Callable[[], None], needs: tuple[str, ...] = ()) -> None:
    """Run a check, or a demo that depends on your code, without derailing the notebook.

    A stub you have not filled in yet simply says so. A wrong answer prints the check's own
    message — which names the likely mistake — and the notebook carries on, so one broken
    exercise never hides the feedback on the others. A demo names the exercises it `needs`:
    until each has passed its check, the demo says which one it is waiting for and skips.
    Nothing is swallowed: every outcome is recorded in `_STATUS` for the progress board at the
    foot of the notebook, and every failure in `_FAILED_CHECKS`, which ends a script run
    non-zero.
    """
    waiting = [name for name in _EXERCISES   # in the order you meet them
               if name in needs and _STATUS.get(name) != "passed"]
    if waiting:
        _STATUS[label] = "not started"
        print(f"{label}: skipped — needs {_named(waiting)} to pass first.")
        return
    try:
        check()
    except NotImplementedError as exc:
        _STATUS[label] = "not started"
        stub = traceback.extract_tb(exc.__traceback__)[-1].name   # the frame that raised
        owner = [name for name, funcs in _EXERCISES.items() if stub in funcs and name != label]
        if owner:
            print(f"{label}: skipped — needs {_named(owner)} first.")
        elif label in _EXERCISES:
            print(f"{label}: not implemented yet — fill in {stub}() above, then re-run "
                  "this cell.")
        else:
            print(f"{label}: skipped — {stub}() is not implemented yet.")
    except AssertionError as exc:
        _STATUS[label] = "failed"
        _FAILED_CHECKS.append(label)
        print(f"{label}: FAILED — {exc}")
    except Exception as exc:  # a half-finished implementation raising something else
        _STATUS[label] = "failed"
        _FAILED_CHECKS.append(label)
        print(f"{label}: raised {type(exc).__name__}: {exc}")
    else:
        _STATUS[label] = "passed"

## What you carry in from modules 1 and 4

This lesson runs alone, in an empty directory, so it cannot import the earlier lessons. It
carries minimal faithful copies of what it reuses instead, under the same names and with
the same conventions: `quantile_edges` and `expected_calibration_error` from module 1,
`average_ranks`, `auc_by_ranks` and `paired_bootstrap_difference` from module 4, and module
1's `challenger_decision`, the promotion rule written down before the numbers are seen.
They are given to you and not graded. Read `challenger_decision` again before exercise 7:
its absolute gates come first, its limits are inclusive, and it reports every rule.

In [ ]:
def _sigmoid(x: np.ndarray) -> np.ndarray:
    return 1.0 / (1.0 + np.exp(-x))


def quantile_edges(x: np.ndarray, n_bins: int) -> np.ndarray:
    """Module 1. Edges at equally spaced quantiles of `x`, outer edges opened to infinity.

    The edges are cut ONCE, on the development sample, and travel with the model.
    """
    edges = np.quantile(np.asarray(x, dtype=float), np.linspace(0.0, 1.0, n_bins + 1))
    edges = np.unique(edges)
    edges[0] = -np.inf
    edges[-1] = np.inf
    return edges


def expected_calibration_error(y_true: np.ndarray, y_prob: np.ndarray,
                               n_bins: int = N_BINS) -> float:
    """Module 1. Support-weighted mean absolute gap between predicted and observed rates.

    Equal-width bins over [0, 1]; a probability on an interior edge belongs to the LOWER bin
    and 0.0 to the first; an empty bin carries no weight; the gap in each bin is absolute.
    """
    y = np.asarray(y_true, dtype=float)
    p = np.asarray(y_prob, dtype=float)
    if y.shape != p.shape or y.size == 0 or n_bins < 1:
        raise ValueError("ECE needs two non-empty arrays of the same length and n_bins >= 1")
    if ((p < 0.0) | (p > 1.0)).any() or not np.isin(y, (0.0, 1.0)).all():
        raise ValueError("probabilities must lie in [0, 1] and labels must be 0 or 1")
    edges = np.linspace(0.0, 1.0, n_bins + 1)
    idx = np.clip(np.searchsorted(edges, p, side="left") - 1, 0, n_bins - 1)
    count = np.bincount(idx, minlength=n_bins)
    sum_p = np.bincount(idx, weights=p, minlength=n_bins)
    sum_y = np.bincount(idx, weights=y, minlength=n_bins)
    occ = count > 0
    gap = np.abs(sum_p[occ] / count[occ] - sum_y[occ] / count[occ])
    return float((count[occ] / p.size * gap).sum())


def average_ranks(x: np.ndarray) -> np.ndarray:
    """Module 4. 1-based ranks in input order, tied values sharing the mean of their ranks."""
    x = np.asarray(x, dtype=float)
    order = np.argsort(x, kind="mergesort")
    _, inverse, counts = np.unique(x[order], return_inverse=True, return_counts=True)
    block_start = np.concatenate(([0], np.cumsum(counts)[:-1]))
    out = np.empty(x.size, dtype=float)
    out[order] = block_start[inverse] + (counts[inverse] - 1) / 2.0 + 1.0
    return out


def auc_by_ranks(y_true: np.ndarray, y_score: np.ndarray) -> float:
    """Module 4. AUC by the Mann-Whitney rank identity, ties counted as half."""
    y = np.asarray(y_true)
    p = np.asarray(y_score, dtype=float)
    if y.shape != p.shape or y.size == 0 or not np.isin(y, (0, 1)).all():
        raise ValueError("AUC needs equal-length labels in {0, 1} and scores")
    n_pos = int((y == 1).sum())
    n_neg = int(y.size - n_pos)
    if n_pos == 0 or n_neg == 0:
        raise ValueError(f"AUC is undefined with {n_pos} events and {n_neg} non-events")
    ranks = average_ranks(p)
    return float((ranks[y == 1].sum() - n_pos * (n_pos + 1) / 2.0) / (n_pos * n_neg))


class PairedDifference(NamedTuple):
    """Module 4. The difference between two models scored on the same records."""

    diff: float             # auc_a - auc_b on the original sample
    lo: float
    hi: float
    se_paired: float        # sd of the replicate DIFFERENCES
    se_independent: float   # sqrt(se_a**2 + se_b**2), the wrong answer, kept for contrast
    corr: float             # correlation of the two models' replicate AUCs
    n_boot: int


def paired_bootstrap_difference(y_true: np.ndarray, score_a: np.ndarray, score_b: np.ndarray,
                                n_boot: int = N_BOOT, alpha: float = ALPHA,
                                rng: np.random.Generator | None = None) -> PairedDifference:
    """Module 4. Percentile interval for AUC(a) - AUC(b) from ONE stratified draw per replicate."""
    y = np.asarray(y_true)
    a = np.asarray(score_a, dtype=float)
    b = np.asarray(score_b, dtype=float)
    if not (y.shape == a.shape == b.shape):
        raise ValueError(f"shapes differ: labels {y.shape}, a {a.shape}, b {b.shape}")
    if n_boot < 2 or not 0.0 < alpha < 1.0:
        raise ValueError("n_boot must be at least 2 and alpha must lie in (0, 1)")
    if rng is None:
        rng = np.random.default_rng(SEED)
    idx_pos = np.flatnonzero(y == 1)
    idx_neg = np.flatnonzero(y == 0)
    reps_a = np.empty(n_boot, dtype=float)
    reps_b = np.empty(n_boot, dtype=float)
    for i in range(n_boot):
        take = np.concatenate([rng.choice(idx_pos, idx_pos.size, replace=True),
                               rng.choice(idx_neg, idx_neg.size, replace=True)])
        y_take = y[take]
        reps_a[i] = auc_by_ranks(y_take, a[take])
        reps_b[i] = auc_by_ranks(y_take, b[take])
    reps_d = reps_a - reps_b
    lo, hi = np.quantile(reps_d, [alpha / 2.0, 1.0 - alpha / 2.0])
    se_a = float(reps_a.std(ddof=1))
    se_b = float(reps_b.std(ddof=1))
    corr = float(np.corrcoef(reps_a, reps_b)[0, 1]) if se_a > 0.0 and se_b > 0.0 else 1.0
    return PairedDifference(
        diff=float(auc_by_ranks(y, a) - auc_by_ranks(y, b)), lo=float(lo), hi=float(hi),
        se_paired=float(reps_d.std(ddof=1)), se_independent=float(math.hypot(se_a, se_b)),
        corr=corr, n_boot=int(n_boot))


class Decision(NamedTuple):
    """Module 1. A verdict, the reason for every rule that produced it, and the margins."""

    verdict: str                 # "promote" | "hold" | "reject"
    reasons: tuple               # one string per rule, in order, each starting PASS or FAIL
    margins: dict                # how far each rule was cleared or missed, by


def challenger_decision(champion: Mapping[str, float], challenger: Mapping[str, float],
                        policy: Mapping[str, float]) -> Decision:
    """Module 1. Absolute gates first, then improvement; inclusive limits; every rule reported."""
    missing = [f"champion.{k}" for k in ("auc", "ece") if k not in champion]
    missing += [f"challenger.{k}" for k in ("auc", "ece") if k not in challenger]
    missing += [f"policy.{k}" for k in ("min_auc", "max_ece", "min_auc_gain") if k not in policy]
    if missing:
        raise ValueError("missing required keys: " + ", ".join(missing))
    gain = float(challenger["auc"]) - float(champion["auc"])
    ece_change = float(challenger["ece"]) - float(champion["ece"])
    tests = (
        (float(challenger["auc"]) >= float(policy["min_auc"]),
         f"challenger AUC {challenger['auc']:.4f} against the floor of {policy['min_auc']:.4f}"),
        (float(challenger["ece"]) <= float(policy["max_ece"]),
         f"challenger ECE {challenger['ece']:.4f} against the ceiling of "
         f"{policy['max_ece']:.4f}"),
        (gain >= float(policy["min_auc_gain"]),
         f"AUC gain {gain:+.4f} against the required {policy['min_auc_gain']:.4f}"),
        (ece_change <= 0.0,
         f"calibration change {ece_change:+.4f} against a ceiling of +0.0000"),
    )
    reasons = tuple(("PASS " if ok else "FAIL ") + text for ok, text in tests)
    if not (tests[0][0] and tests[1][0]):
        verdict = "reject"
    elif tests[2][0] and tests[3][0]:
        verdict = "promote"
    else:
        verdict = "hold"
    margins = {"auc_gain": gain, "ece_change": ece_change,
               "auc_headroom": float(challenger["auc"]) - float(policy["min_auc"]),
               "ece_headroom": float(policy["max_ece"]) - float(challenger["ece"])}
    return Decision(verdict=verdict, reasons=reasons, margins=margins)

## 1. What you have been handed

Below is the generator. Read it — the one advantage a synthetic book gives you is that you
may know exactly how the world you are modelling works. Four things in it matter later:

- **`income` is missing for thin files**, and a thin file is riskier than its other inputs
  say. How a pipeline treats that missing value is not a detail.
- **`collections_flag`** is set when an account is passed to collections, which happens
  only after it has defaulted. It sits in the development extract beside the real inputs.
- **the champion** is a documented logistic model, written into the code below as the
  model inventory would record it, including its rule for a missing income.
- **the vendor's score** arrives as a finished file, sorted by the vendor's own score.

Two extracts reach the validator before any model is built: the first line's scored file
for the champion, and the vendor's.

In [ ]:
def synthetic_book(rng: np.random.Generator, n: int, first_id: int,
                   months: Sequence[str]) -> dict:
    """One deterministic synthetic credit book. SYNTHETIC — see DATA_NOTE."""
    z = rng.normal(0.0, 1.0, n)                                   # the risk nobody observes
    utilisation = np.clip(_sigmoid(0.9 * z + rng.normal(0.0, 0.9, n) + 0.2), 0.0, 1.0).round(3)
    thin_file = rng.random(n) < _sigmoid(-2.1 + 0.6 * z)
    income = np.round(np.exp(10.5 - 0.30 * z + rng.normal(0.0, 0.45, n)), -1)
    income[thin_file] = np.nan
    delinquencies = np.minimum(rng.poisson(np.exp(-1.3 + 0.55 * z)), 6).astype(float)
    months_on_book = np.clip(np.round(np.exp(3.9 - 0.25 * z + rng.normal(0.0, 0.55, n))),
                             3.0, 360.0)
    true_logit = -2.2 + 1.0 * z + 0.7 * thin_file - 0.45 * (np.log(months_on_book) - 3.9)
    y = (rng.random(n) < _sigmoid(true_logit)).astype(np.int64)
    collections_flag = ((y == 1) & (rng.random(n) < 0.30)).astype(float)   # AFTER default
    period = np.asarray(months)[rng.integers(0, len(months), n)]
    # The vendor sees the risk more sharply than anyone else, and is over-confident about it.
    vendor = _sigmoid(1.6 * (true_logit + rng.normal(0.0, 0.45, n)) + 1.0)
    return {"record_id": first_id + np.arange(n, dtype=np.int64), "period": period,
            "utilisation": utilisation, "income": income, "delinquencies": delinquencies,
            "months_on_book": months_on_book, "collections_flag": collections_flag,
            "y": y, "vendor": vendor}


# The champion, as the model inventory documents it: champion-v3, a logistic model. A missing
# income takes CHAMPION_INCOME_FILL inside the log AND switches income_missing on.
CHAMPION_SPEC = {"intercept": 3.70, "utilisation": 2.25, "log_income": -0.67,
                 "income_missing": 1.16, "delinquencies": 0.23}
CHAMPION_INCOME_FILL = 36000.0


def champion_model(inputs: Mapping[str, np.ndarray]) -> np.ndarray:
    """champion-v3's documented scoring rule, applied to inputs as prepared (NaN = missing)."""
    income = np.asarray(inputs["income"], dtype=float)
    missing = np.isnan(income)
    logit = (CHAMPION_SPEC["intercept"]
             + CHAMPION_SPEC["utilisation"] * inputs["utilisation"]
             + CHAMPION_SPEC["log_income"] * np.log(np.where(missing, CHAMPION_INCOME_FILL, income))
             + CHAMPION_SPEC["income_missing"] * missing
             + CHAMPION_SPEC["delinquencies"] * inputs["delinquencies"])
    return _sigmoid(logit)


class EvalFrame(NamedTuple):
    """What one model was judged on: which records, from when, with which inputs, and its scores."""

    model: str
    record_id: np.ndarray    # int, one per record
    period: np.ndarray       # "YYYY-MM" the inputs were taken, per record
    inputs: dict             # name -> float array AS DELIVERED to the model (NaN = left missing)
    y: np.ndarray            # the outcome the model is judged against, 0 or 1
    score: np.ndarray        # the model's probability of the event


def make_frame(model: str, book: Mapping[str, np.ndarray], score: np.ndarray,
               inputs: Mapping[str, np.ndarray] | None = None) -> EvalFrame:
    """Package a book and a model's scores on it. `inputs` defaults to the book's own."""
    use = {k: np.asarray(book[k], dtype=float) for k in INPUTS} if inputs is None else dict(inputs)
    return EvalFrame(model, np.asarray(book["record_id"]), np.asarray(book["period"]), use,
                     np.asarray(book["y"]), np.asarray(score, dtype=float))


_rng = np.random.default_rng(SEED)
DEV = synthetic_book(_rng, N_DEV, 100001, DEV_MONTHS)
VAL = synthetic_book(_rng, N_VAL, 200001, VAL_MONTHS)

# The first line's pipeline filled every missing income before the champion saw it.
_first_line_inputs = {k: np.asarray(VAL[k], dtype=float).copy() for k in INPUTS}
_first_line_inputs["income"] = np.where(np.isnan(VAL["income"]), CHAMPION_INCOME_FILL,
                                        VAL["income"])
CHAMPION_EXTRACT = make_frame("champion-v3 (first-line extract)", VAL,
                              champion_model(_first_line_inputs), _first_line_inputs)
# The vendor's file: the same records, returned sorted by its own score, highest first.
_vendor_order = np.argsort(-VAL["vendor"], kind="mergesort")
VENDOR_FRAME = make_frame("vendor-challenger", {k: np.asarray(v)[_vendor_order]
                                                for k, v in VAL.items()},
                          VAL["vendor"][_vendor_order])

print(f"development book: {N_DEV} records, {DEV_MONTHS[0]} to {DEV_MONTHS[-1]}, "
      f"event rate {DEV['y'].mean():.4f}, income missing on {int(np.isnan(DEV['income']).sum())}")
print(f"validation book:  {N_VAL} records, {VAL_MONTHS[0]} to {VAL_MONTHS[-1]}, "
      f"event rate {VAL['y'].mean():.4f}, income missing on {int(np.isnan(VAL['income']).sum())}")
for _frame in (CHAMPION_EXTRACT, VENDOR_FRAME):
    print(f"  {_frame.model:<34} AUC {auc_by_ranks(_frame.y, _frame.score):.4f}  "
          f"ECE {expected_calibration_error(_frame.y, _frame.score):.4f}")
print("\n" + DATA_NOTE)

## 2. Exercise 1 — `design_matrix()`

Logistic regression models the log odds of the event as a linear function of the inputs,
`log(p / (1 - p)) = X b`, and everything IRLS does happens to the matrix `X`. Its first
column is a column of ones — the intercept — and every other column is an input, already
prepared. *Already* is the point: a design matrix has no missing values in it, because
their treatment is a modelling decision taken, and written down, before the matrix exists.

Two things make the weighted normal equations of exercise 2 unsolvable, and one of them is
yours to stop here: a constant column is the intercept a second time.

<details><summary>💡 Hint 1 — what to think about</summary>

The graded decisions are about refusing bad input with a message a model developer can act
on. Which column was it, and why can it not go in? A missing value is a treatment not yet
decided; a constant column duplicates the intercept. The order of the columns is part of
the answer, because every coefficient is later read off by position.

</details>
<details><summary>💡 Hint 2 — the approach in words</summary>

Refuse an empty mapping and a column literally called intercept. Walk the columns in the
order given, converting each to a float array; check it is one-dimensional, as long as the
first, free of NaN and infinity, and not everywhere equal to its own first value — naming
the column in every error. Then stack a column of ones in front of them, and return the
matrix with the names, intercept first.

</details>

In [ ]:
class DesignMatrix(NamedTuple):
    """The matrix a logistic regression is fitted on, and the name of every column."""

    X: np.ndarray       # (n, 1 + k) float64; column 0 is the intercept, all ones
    names: tuple        # ("intercept", *the k column names, in the order given)


def design_matrix(columns: Mapping[str, np.ndarray]) -> DesignMatrix:
    """Stack named 1-D columns behind an intercept column of ones.

    Worked example:

        >>> d = design_matrix({"utilisation": np.array([0.2, 0.9, 0.5])})
        >>> d.names
        ('intercept', 'utilisation')
        >>> d.X
        array([[1. , 0.2],
               [1. , 0.9],
               [1. , 0.5]])

    Requirements, each graded:
      * column 0 is the intercept, a column of ones named "intercept"; the other columns
        follow in the order the mapping gives them, and `X` is float64.
      * ValueError NAMING THE COLUMN if any of its values is missing (NaN) or infinite. A
        design matrix has no missing values: their treatment is decided before it.
      * ValueError naming the column if it is constant. A constant column is the intercept
        again, and X'WX cannot be solved with the same column in it twice.
      * ValueError if the columns differ in length, if a column is not 1-D, if there are no
        columns at all, or if one of them is itself called "intercept".

    Returns:
        DesignMatrix(X, names) — X of shape (n, 1 + number of columns).
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_design_matrix() -> None:
    d = design_matrix({"utilisation": np.array([0.2, 0.9, 0.5]),
                       "delinquencies": np.array([0, 2, 1])})
    assert isinstance(d, DesignMatrix), "return a DesignMatrix(X=..., names=...), not a bare array"
    assert d.names == ("intercept", "utilisation", "delinquencies"), (
        f"names {d.names} — the intercept comes first, then the columns in the order given"
    )
    assert d.X.shape == (3, 3) and d.X.dtype == np.float64, (
        f"X has shape {d.X.shape} and dtype {d.X.dtype}; expected (3, 3) float64"
    )
    assert np.all(d.X[:, 0] == 1.0), "column 0 is the intercept: a column of ones"
    assert np.allclose(d.X[:, 2], [0, 2, 1]), "column order must follow the mapping's order"
    for bad, why, name in (
        ({"income": np.array([1.0, np.nan, 3.0])}, "a missing value", "income"),
        ({"utilisation": np.array([0.4, 0.4, 0.4])}, "a constant column", "utilisation"),
    ):
        try:
            design_matrix(bad)
        except ValueError as exc:
            assert name in str(exc), (
                f"{why} raised ValueError, but the message does not name the column {name!r}: "
                f"{exc} — a developer has to know WHICH column to fix"
            )
        else:
            raise AssertionError(f"{why} must raise ValueError, not reach the fitter")
    try:
        design_matrix({"a": np.array([1.0, 2.0]), "b": np.array([1.0, 2.0, 3.0])})
    except ValueError:
        pass
    else:
        raise AssertionError("columns of different lengths must raise ValueError")
    print("exercise 1 looks right")

In [ ]:
_try("exercise 1", _check_design_matrix)

## 3. Exercise 2 — `fit_logistic_irls()`

There is no closed form for the logistic maximum-likelihood estimate in general, so it is
found by iterating. Each IRLS update linearises the model around the current coefficients
and solves a weighted least-squares problem:

> `p = sigmoid(X b)`, `w = p (1 - p)`, `z = X b + (y - p) / w`, and `(X' W X) b_new = X' W z`

which is algebraically the Newton step `b_new = b + solve(X' W X, X' (y - p))`. Solve the
system; never invert `X' W X`. An inverse costs more, is less accurate, and turns a
near-singular warning into a silently wrong answer.

"Converged" is not a feeling. This lesson's criterion: stop when no coefficient moved by
more than `TOL` in the last update, and give up after `MAX_ITER` updates. Run the cell to
see the stated values; R's `glm` uses the same maximum by default, with a different
criterion that section 3's second demonstration puts on trial.

In [ ]:
print(f"convergence: largest coefficient step <= {TOL:g}; at most {MAX_ITER} updates")
print(f"quasi-complete separation: a probability of the observed outcome >= 1 - {SEPARATION_EPS:g}")

**Separation is the case where there is no answer.** If some combination of the inputs
puts every event above every non-event, doubling the coefficients always fits better, so
the likelihood has no maximum and IRLS walks off towards infinity by roughly the same step
each update. That is complete separation. Quasi-complete separation is the same thing for a
subset: one cell of the book is predicted perfectly, and its coefficient runs away while the
others settle. Any method that maximises the likelihood must end in a separating solution
if one exists — SAS's documentation builds its own check on that fact — so the fitter can
detect it rather than report it.

The two rules you implement are stated in full in the docstring. The first is certain: if
the current linear predictor ranks every event strictly above every non-event, the current
coefficients ARE a separating direction. The second is the quasi-complete fallback.

<details><summary>💡 Hint 1 — what to think about</summary>

Three outcomes, and each one has a different thing it is allowed to return. Think about
when in the loop each rule is tested, and on which coefficients: the separation test and
the step test both look at the update just made. Think about what a reader of `coef` can
do with a number that has no finite true value — and what `path` is for instead. And think
about units: a coefficient's size depends on the unit of its column.

</details>
<details><summary>💡 Hint 2 — the approach in words</summary>

Validate the labels and both classes first. Start from zeros. In each update compute the
fitted probabilities and weights, form X'WX and the right-hand side, solve, and record the
new coefficients. After the update, test the new linear predictor: the smallest value among
the events above the largest among the non-events means separation; otherwise a largest
absolute step within the tolerance means convergence. If the loop runs out, look at each
record's fitted probability of its own outcome. For a separation finding, scale each
non-intercept coefficient by its column's standard deviation and name the largest.

</details>

In [ ]:
class LogisticFit(NamedTuple):
    """The outcome of one IRLS run. `coef` exists only when there is an estimate to report."""

    status: str                # "converged" | "not converged" | "separation"
    coef: np.ndarray | None    # the maximum-likelihood estimates; None unless "converged"
    names: tuple               # the design matrix's column names, intercept first
    n_iter: int                # IRLS updates performed
    path: np.ndarray           # (n_iter, k): coefficients after each update — diagnostics only
    finding: str               # one line for a findings register, opening with the status


def fit_logistic_irls(design: DesignMatrix, y: np.ndarray, tol: float = TOL,
                      max_iter: int = MAX_ITER) -> LogisticFit:
    """Logistic regression by iteratively reweighted least squares, starting from b = 0.

    One update, from coefficients b:
        eta = X @ b ;  p = 1 / (1 + exp(-eta)) ;  w = p * (1 - p)
        solve (X' W X) b_new = X' W z,  z = eta + (y - p) / w
    — the same step as  b_new = b + solve(X' W X, X' (y - p)).

    Worked example — an intercept-only model has a closed form, the log odds of the sample:
        X = [[1], [1], [1], [1]], y = [0, 0, 0, 1]
        -> status "converged", coef = [ln(1/3)] = [-1.0986...]

    The rules, tested on the NEW coefficients after every update, in this order:
      1. complete separation — if the new linear predictor puts every event strictly above
         every non-event (an in-sample AUC of exactly 1), stop: status "separation".
      2. convergence — if no coefficient changed by more than `tol` in that update, stop:
         status "converged".
    and once, if all `max_iter` updates ran without either:
      3. quasi-complete separation — if any record's fitted probability OF ITS OWN OUTCOME
         is at least 1 - SEPARATION_EPS: status "separation"; otherwise "not converged".

    Requirements, each graded:
      * solve the linear system (np.linalg.solve or lstsq). Never form an explicit inverse:
        the rubric fails a call to np.linalg.inv or np.linalg.pinv.
      * `coef` is an array only when status is "converged", and None otherwise. A coefficient
        with a warning attached is a number that ends up in a spreadsheet without its warning.
      * `path` holds the coefficients after each update, shape (n_iter, k); when converged,
        its last row is `coef`. `n_iter` counts the updates performed.
      * `finding` opens with "CONVERGED", "NOT CONVERGED" or "SEPARATION". A separation finding
        names the column that runs away: the non-intercept column whose STANDARDISED
        coefficient |b_j| * std(X[:, j]) is largest at the last update. A raw |b_j| depends on
        the column's units and can point at the wrong variable.
      * ValueError if y is not a 1-D array of 0s and 1s as long as X, or if a class is absent.

    Returns:
        LogisticFit(status, coef, names, n_iter, path, finding).
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_irls() -> None:
    one = fit_logistic_irls(DesignMatrix(np.ones((4, 1)), ("intercept",)), np.array([0, 0, 0, 1]))
    assert isinstance(one, LogisticFit), "return a LogisticFit(status=..., coef=..., ...)"
    assert one.status == "converged" and one.coef is not None, (
        f"an intercept-only model on [0, 0, 0, 1] converges; got status {one.status!r}"
    )
    assert abs(one.coef[0] - math.log(1 / 3)) < 1e-8, (
        f"the intercept-only fit should be ln(1/3) = {math.log(1 / 3):.6f}, got "
        f"{one.coef[0]:.6f} — check the sign of (y - p) in the right-hand side"
    )
    assert one.path.shape == (one.n_iter, 1) and np.allclose(one.path[-1], one.coef), (
        f"path must have one row per update, shape ({one.n_iter}, 1), ending at coef"
    )
    assert one.finding.startswith("CONVERGED"), f"finding should open 'CONVERGED': {one.finding!r}"
    x = np.array([-3.0, -2.0, -1.0, 1.0, 2.0, 3.0])
    sep = fit_logistic_irls(design_matrix({"x": x}), np.array([0, 0, 0, 1, 1, 1]))
    assert sep.status == "separation", (
        f"x separates the classes perfectly, got status {sep.status!r} — test whether the new "
        "linear predictor puts every event above every non-event after each update"
    )
    assert sep.coef is None, (
        "a separated fit must return coef=None: there is no finite estimate, and a number "
        "with a flag attached gets copied without its flag"
    )
    assert sep.n_iter < MAX_ITER, (
        f"separation was declared only after {sep.n_iter} updates, the whole budget — rule 1 "
        "is certain as soon as the new linear predictor orders the classes; stop there"
    )
    assert sep.finding.startswith("SEPARATION") and "x" in sep.finding, (
        f"the separation finding must open 'SEPARATION' and name the column: {sep.finding!r}"
    )
    rng = np.random.default_rng(7)
    u = rng.random(300)
    yy = (rng.random(300) < _sigmoid(-1.0 + 2.0 * u)).astype(np.int64)
    ok = fit_logistic_irls(design_matrix({"u": u}), yy)
    assert ok.status == "converged" and ok.n_iter <= 12, (
        f"a well-posed fit should converge in a handful of updates, got {ok.status!r} after "
        f"{ok.n_iter} — is the step X'(y - p) solved against X'WX, not X'X?"
    )
    short = fit_logistic_irls(design_matrix({"u": u}), yy, max_iter=1)
    assert short.status == "not converged" and short.coef is None, (
        f"one update cannot meet a 1e-8 tolerance: expected 'not converged' with coef None, "
        f"got {short.status!r}"
    )
    print("exercise 2 looks right")

In [ ]:
_try("exercise 2", _check_irls)

Now fit the development book. The candidate list is every field in the extract that could
plausibly be a predictor, `collections_flag` included, and the challenger's documented
treatment of a missing income is the development median inside the log plus an indicator.
Run it and read the finding before you read the coefficients.

In [ ]:
CANDIDATES = ("utilisation", "log_income", "income_missing", "delinquencies",
              "log_months_on_book", "collections_flag")
CHALLENGER_FEATURES = CANDIDATES[:-1]
DEV_INCOME_MEDIAN = float(np.nanmedian(DEV["income"]))


def challenger_columns(inputs: Mapping[str, np.ndarray], names: Sequence[str],
                       income_fill: float = DEV_INCOME_MEDIAN) -> dict:
    """The IRLS challenger's documented preparation: a missing income takes the development
    median inside the log, and income_missing records that it did."""
    income = np.asarray(inputs["income"], dtype=float)
    missing = np.isnan(income)
    prepared = {"utilisation": inputs["utilisation"],
                "log_income": np.log(np.where(missing, income_fill, income)),
                "income_missing": missing.astype(float),
                "delinquencies": inputs["delinquencies"],
                "log_months_on_book": np.log(inputs["months_on_book"])}
    if "collections_flag" in inputs:
        prepared["collections_flag"] = inputs["collections_flag"]
    return {name: prepared[name] for name in names}


_BUILT: dict = {}     # what the demonstrations build, for the ones after them


def _show_separation() -> None:
    everything = design_matrix(challenger_columns(DEV, CANDIDATES))
    first = fit_logistic_irls(everything, DEV["y"])
    print(f"all candidates: {first.finding}")
    flag = DEV["collections_flag"] == 1
    print(f"\n  collections_flag = 1: {int(DEV['y'][flag].sum()):>4} defaults, "
          f"{int((DEV['y'][flag] == 0).sum()):>4} non-defaults")
    print(f"  collections_flag = 0: {int(DEV['y'][~flag].sum()):>4} defaults, "
          f"{int((DEV['y'][~flag] == 0).sum()):>4} non-defaults")
    print("  An empty cell. The flag is set after default, so it predicts the past perfectly.")
    chosen = design_matrix(challenger_columns(DEV, CHALLENGER_FEATURES))
    fit = fit_logistic_irls(chosen, DEV["y"])
    print(f"\nwithout it: {fit.finding}")
    for name, value in zip(fit.names, fit.coef):
        print(f"  {name:<20} {value:+.4f}")
    _BUILT["separated"] = (everything, first)
    _BUILT["irls"] = (chosen, fit)


_try("the development fit", _show_separation, needs=("exercise 1", "exercise 2"))

Your fitter refused to report `collections_flag`. A fitter with a weaker test would not
have. R's `glm` stops by default when the deviance stops moving, `|dev - dev_old| / (|dev| +
0.1) < 1e-8`. Run that rule over your own separated fit's `path`:

In [ ]:
def _built(*keys: str) -> bool:
    """True when the demonstrations these keys come from have run; otherwise say which did not."""
    absent = [k for k in keys if k not in _BUILT]
    if absent:
        print("skipped — an earlier demonstration did not finish (" + ", ".join(absent)
              + "); re-run the cells above it.")
    return not absent


def _show_deviance_rule() -> None:
    if not _built("separated"):
        return
    design, fit = _BUILT["separated"]
    X, y = design.X, DEV["y"].astype(float)
    j = fit.names.index("collections_flag")

    def deviance(b: np.ndarray) -> float:
        eta = X @ b
        return float(-2.0 * (y * eta - np.logaddexp(0.0, eta)).sum())

    previous, stopped = deviance(np.zeros(X.shape[1])), None
    for k, b in enumerate(fit.path, start=1):
        current = deviance(b)
        if stopped is None and abs(current - previous) / (abs(current) + 0.1) < 1e-8:
            stopped = k
        previous = current
    for k in range(4, fit.n_iter + 1, 4):
        print(f"  after update {k:>2}: collections_flag {fit.path[k - 1][j]:7.3f}, "
              f"deviance {deviance(fit.path[k - 1]):.6f}")
    if stopped is None:
        print("the deviance rule never fired inside the iteration limit")
    else:
        print(f"\nthe deviance rule would have declared convergence at update {stopped}, and "
              f"reported collections_flag = {fit.path[stopped - 1][j]:.2f}")
        print("The deviance had stopped moving. The coefficient had not: it gains about the "
              "same\namount every update, forever. That number would have gone into a "
              "validation report.")


_try("the deviance rule", _show_deviance_rule, needs=("exercise 1", "exercise 2"))

## 4. Exercise 3 — prove the fit is the maximum

With no reference library to agree with, a hand-built fitter needs two proofs that it found
the maximum-likelihood estimate.

**The score equations.** The gradient of the log-likelihood is `X'(y - p)`. At the maximum
it is zero — every entry — because that is what a maximum is. Its first entry, for the
intercept, says more: the fitted probabilities sum to the number of events, so a fitted
logistic model is calibrated in the large on its own development sample, by construction.

**A closed form.** With one binary predictor the model has two parameters and the 2×2 table
has two odds, so the fit reproduces the table exactly: the intercept is the log odds where
`x = 0`, and the slope is the log odds ratio. A zero cell makes one of those odds 0 or
infinite — which is quasi-complete separation in its smallest form, and why
`closed_form_2x2` returns `None` for it, exactly as your fitter reports no coefficient.

<details><summary>💡 Hint 1 — what to think about</summary>

The score is a gradient, not a residual: one number per column, each the sum over records
of that column times the prediction error. For the closed form, think about which records
land in which cell, which ratio of cells is an odds, and which ratio of two odds is the
slope. Then think about what a logarithm of zero would do to a report.

</details>
<details><summary>💡 Hint 2 — the approach in words</summary>

For the score: check that the coefficient vector has one entry per column and the labels one
per row, turn the linear predictor into probabilities, and multiply the transposed matrix
into the label-minus-probability vector. For the closed form: validate both arrays as binary
and equal in length, with a contrast in x and both classes in y; count the four cells; return
None if any is zero; otherwise take the log of the event odds in the x-equals-zero row, and
the log of the cross-product ratio, as plain floats.

</details>

In [ ]:
def score_equations(design: DesignMatrix, y: np.ndarray, coef: np.ndarray) -> np.ndarray:
    """The score vector X'(y - p), the log-likelihood's gradient at `coef`.

    Worked example:
        X = [[1, 0], [1, 1]], y = [0, 1], coef = [0, 0]   ->   p = [0.5, 0.5]
        X'(y - p) = [(-0.5) + (0.5), 0 * (-0.5) + 1 * (0.5)] = [0.0, 0.5]

    Requirements, each graded:
      * one entry per column of X, as a float array
      * ValueError if `coef` does not have one entry per column, or `y` one entry per row

    Returns:
        np.ndarray of shape (k,) — zero in every entry at the maximum-likelihood estimate.
    """
    # YOUR CODE HERE
    raise NotImplementedError


class TwoByTwo(NamedTuple):
    """The maximum-likelihood logistic fit of y on one binary x, by counting."""

    intercept: float    # ln(n01 / n00): the log odds of the event where x == 0
    slope: float        # ln(n11 * n00 / (n10 * n01)): the log odds ratio


def closed_form_2x2(x: np.ndarray, y: np.ndarray) -> TwoByTwo | None:
    """The logistic fit of y on a single binary x, read straight off the 2×2 table.

    Cells: n_ab is the number of records with x == a and y == b.

    Worked example:
        x = [0, 0, 0, 0, 1, 1, 1, 1, 1]
        y = [0, 0, 0, 1, 0, 1, 1, 1, 1]
        n00 = 3, n01 = 1, n10 = 1, n11 = 4
        intercept = ln(1 / 3) = -1.0986...,  slope = ln(4 * 3 / (1 * 1)) = ln 12 = 2.4849...

    Requirements, each graded:
      * None when any cell is zero: one of the two odds is 0 or infinite, and so is the slope.
      * plain Python floats in the result
      * ValueError if x or y hold anything but 0 and 1, if they differ in length, if x is
        constant (there is no contrast to estimate), or if either class of y is absent

    Returns:
        TwoByTwo(intercept, slope), or None when a cell is empty.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_verification() -> None:
    d = DesignMatrix(np.array([[1.0, 0.0], [1.0, 1.0]]), ("intercept", "x"))
    s = score_equations(d, np.array([0, 1]), np.array([0.0, 0.0]))
    assert np.allclose(s, [0.0, 0.5]), (
        f"the worked example gives [0.0, 0.5], got {np.round(s, 6)} — the score is X'(y - p), "
        "a vector with one entry per column"
    )
    t = closed_form_2x2(np.array([0, 0, 0, 0, 1, 1, 1, 1, 1]),
                        np.array([0, 0, 0, 1, 0, 1, 1, 1, 1]))
    assert t is not None and abs(t.intercept - math.log(1 / 3)) < 1e-12, (
        f"the worked example's intercept is ln(1/3), got {t} — it is the log odds where x == 0"
    )
    assert abs(t.slope - math.log(12)) < 1e-12, (
        f"the worked example's slope is ln 12, got {t.slope} — the cross-product "
        "(n11 * n00) / (n10 * n01), not a difference of rates"
    )
    empty = closed_form_2x2(np.array([0, 0, 1, 1]), np.array([0, 1, 1, 1]))
    assert empty is None, (
        f"a table with a zero cell has no finite slope and must return None, got {empty}"
    )
    print("exercise 3 looks right")

In [ ]:
_try("exercise 3", _check_verification)

In [ ]:
def _show_verification() -> None:
    if not _built("irls"):
        return
    design, fit = _BUILT["irls"]
    score = score_equations(design, DEV["y"], fit.coef)
    bound = 1e-9 * N_DEV
    print(f"score equations at the fitted coefficients: every |entry| below {bound:g}? "
          f"{bool(np.all(np.abs(score) < bound))}")
    print(f"  intercept entry: fitted probabilities sum to the events? "
          f"{bool(abs(score[0]) < bound)}  ({int(DEV['y'].sum())} events)")
    miss = np.isnan(DEV["income"]).astype(np.int64)
    closed = closed_form_2x2(miss, DEV["y"])
    single = fit_logistic_irls(design_matrix({"income_missing": miss}), DEV["y"])
    print(f"\nincome_missing alone, closed form:  intercept {closed.intercept:+.6f}  "
          f"slope {closed.slope:+.6f}")
    print(f"income_missing alone, your IRLS:    intercept {single.coef[0]:+.6f}  "
          f"slope {single.coef[1]:+.6f}")
    print(f"agree to 1e-8: {bool(np.allclose(single.coef, closed, atol=1e-8, rtol=0.0))}")
    flag = DEV["collections_flag"].astype(np.int64)
    print(f"collections_flag alone, closed form: {closed_form_2x2(flag, DEV['y'])}")


_try("the fit, verified", _show_verification, needs=("exercise 1", "exercise 2", "exercise 3"))

## 5. Exercise 4 — weight of evidence

The second challenger is deliberately cruder: a scorecard. Each input is cut into a few
bins, each bin is replaced by one number — its weight of evidence — and a logistic
regression is fitted on those numbers. It cannot bend where the data does not, it gives a
missing value a bin of its own instead of an imputed value, and every score it produces
can be read back to a bin a person can name. That is what makes it conservative.

This lesson's weight of evidence of a bin compares its share of the non-events with its
share of the events, on the log scale; positive means safer than the book as a whole. The
sign is a convention, and a scorecard's documentation must state which one it uses. A bin
with no events would divide by zero, so a smoothing constant is added to both classes of
every occupied bin — and an empty bin reports `nan`, as module 1's reliability table did:
no records, no evidence.

<details><summary>💡 Hint 1 — what to think about</summary>

Two functions: one turns counts into weights, the other turns values into counts. For the
weights, decide which bins are occupied before anything else, because only they are smoothed
and only they count towards the denominators. For the counts, look hard at a value that
sits exactly on an edge, and at a value that is missing: a missing value is not a small
number and must not be binned as one.

</details>
<details><summary>💡 Hint 2 — the approach in words</summary>

For the weights: validate, find the occupied bins, count them, and for those bins divide
each class's smoothed count by that class's total plus the smoothing times the number of
occupied bins; the log of the non-event share over the event share is the weight, and every
other bin stays NaN. For the table: validate the edges and labels, split off the NaNs, place
every other value with a sorted search that sends a value on an edge to the upper bin,
clip into range, count each class per bin, append the missing bin's counts, and weigh.

</details>

In [ ]:
def woe_from_counts(events: np.ndarray, non_events: np.ndarray,
                    smoothing: float = WOE_SMOOTHING) -> np.ndarray:
    """Weight of evidence per bin, from its event and non-event counts.

    For every OCCUPIED bin i, with a = smoothing and m = the number of occupied bins:

        WoE_i = ln( ((non_events_i + a) / (N_non + a*m)) / ((events_i + a) / (N_events + a*m)) )

    An empty bin (no records of either class) reports nan and is not counted in m.

    Worked example (a = 0.5, three occupied bins, so both denominators are 3 + 1.5 = 4.5):
        events = [0, 1, 2], non_events = [2, 1, 0]
        WoE = [ln(2.5 / 0.5), ln(1.5 / 1.5), ln(0.5 / 2.5)] = [1.6094..., 0.0, -1.6094...]

    Requirements, each graded:
      * the formula above, on occupied bins only; nan for an empty bin
      * ValueError if the two arrays differ in length, a count is negative, smoothing is not
        positive, or either class has no records at all

    Returns:
        float array of the same length as the counts.
    """
    # YOUR CODE HERE
    raise NotImplementedError


class WoeTable(NamedTuple):
    """One variable's bins. The last entry of each array is the MISSING bin."""

    edges: np.ndarray       # the value-bin edges, cut once on the development sample
    events: np.ndarray      # int, per value bin in order, then the missing bin
    non_events: np.ndarray  # same layout
    woe: np.ndarray         # same layout


def woe_table(x: np.ndarray, y: np.ndarray, edges: np.ndarray,
              smoothing: float = WOE_SMOOTHING) -> WoeTable:
    """Bin x on `edges`, give missing values a bin of their own, and weigh every bin.

    Value bin i is [edges[i], edges[i+1]): a value on an interior edge belongs to the UPPER
    bin, as in module 1's PSI, and a value beyond the outer edges falls into the end bin. A
    NaN is a missing value and goes to the last bin, which is always present, even empty.

    Worked example:
        x = [1, 2, 3, 4, nan, nan], y = [0, 0, 1, 0, 1, 1], edges = [-inf, 2.5, inf]
        events     = [0, 1, 2]        (bin [-inf, 2.5), bin [2.5, inf), missing)
        non_events = [2, 1, 0]
        woe        = [1.6094..., 0.0, -1.6094...]      (the woe_from_counts example)

    Requirements, each graded:
      * len(edges) - 1 value bins, then the missing bin, in every array
      * a value on an interior edge counts in the upper bin; values past the ends are kept
      * woe from woe_from_counts, with the same smoothing
      * ValueError if `edges` has fewer than two values or is not strictly increasing, if x
        and y differ in length, or if y holds anything but 0 and 1

    Returns:
        WoeTable(edges, events, non_events, woe).
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_woe() -> None:
    w = woe_from_counts(np.array([0, 1, 2]), np.array([2, 1, 0]))
    assert np.allclose(w, [math.log(5), 0.0, -math.log(5)]), (
        f"the worked example gives [ln 5, 0, -ln 5], got {np.round(w, 4)} — add the "
        "smoothing to both classes of every occupied bin, and to the denominators a*m"
    )
    gap = woe_from_counts(np.array([3, 0, 1]), np.array([1, 0, 3]))
    assert np.isnan(gap[1]) and np.isfinite(gap[0]), (
        f"an empty bin reports nan and takes no part in m, got {gap} — no records, no evidence"
    )
    t = woe_table(np.array([1, 2, 3, 4, np.nan, np.nan]), np.array([0, 0, 1, 0, 1, 1]),
                  np.array([-np.inf, 2.5, np.inf]))
    assert t.events.tolist() == [0, 1, 2] and t.non_events.tolist() == [2, 1, 0], (
        f"events {t.events.tolist()}, non_events {t.non_events.tolist()} — two value bins, "
        "then the missing bin LAST; a NaN is not a small number"
    )
    edge = woe_table(np.array([1.0, 2.5, 4.0]), np.array([0, 1, 1]),
                     np.array([-np.inf, 2.5, np.inf]))
    assert edge.events.tolist() == [0, 2, 0], (
        f"2.5 sits on the edge and belongs to the UPPER bin [2.5, inf), got events "
        f"{edge.events.tolist()} — searchsorted with side='right', minus one"
    )
    print("exercise 4 looks right")

In [ ]:
_try("exercise 4", _check_woe)

## 6. Exercise 5 — `enforce_monotonic()`

Conceptual soundness (module 3) documented a direction for every input: risk rises with
utilisation and delinquencies, and falls with income and with time on book. A scorecard
whose bins zig-zag against that direction is fitting noise, and will be asked about it. The
rule this lesson states, and enforces, is to **pool adjacent violators**: while two adjacent
value bins have event rates that move against the documented direction, merge them, and
look again — the merged bin can now violate the direction with its other neighbour.
Whichever violating pair you pool first, the result is the same.

The missing bin is not on the axis, so it takes no part. But pooling changes how many bins
are occupied, and so every bin's smoothed share, the missing bin's included: recompute them
all.

<details><summary>💡 Hint 1 — what to think about</summary>

A single pass that merges each violating pair once is not enough: a merged bin carries a new
rate and must be compared with the bin before it. Equal rates are not a violation. Merging
bins means adding counts and forgetting the edge between them — but never the outer edges.
And an empty value bin has no rate to compare, which is a reason to stop, not to guess.

</details>
<details><summary>💡 Hint 2 — the approach in words</summary>

Validate the direction and refuse an empty value bin. Keep a list of blocks, each holding its
event count, non-event count and the index of the edge it ends on. Push the value bins one at
a time; after each push, while the last two blocks violate the direction, merge them into one.
At the end rebuild the edges from the first edge and each block's closing edge, append the
untouched missing-bin counts, and recompute every weight with the exercise 4 function.

</details>

In [ ]:
def enforce_monotonic(table: WoeTable, direction: str,
                      smoothing: float = WOE_SMOOTHING) -> WoeTable:
    """Pool adjacent value bins until the event rate moves only in the documented direction.

    `direction` is the documented direction of RISK: "increasing" means the event rate must
    never fall from one value bin to the next, "decreasing" that it must never rise. Equal
    rates are not a violation.

    Worked example ("increasing", 100 records in each of four value bins):
        event rates 0.10, 0.26, 0.30, 0.12
        0.30 then 0.12 falls: pool them -> 0.21
        but now 0.26 then 0.21 falls: pool again -> (26 + 30 + 12) / 300 = 0.2267
        result: two value bins, rates 0.10 and 0.2267; the middle two edges are gone

    Requirements, each graded:
      * the missing bin (the last entry) takes no part in pooling and keeps its counts
      * counts are conserved, and the result moves only in `direction`
      * `edges` keeps both outer edges and only the edges between pooled blocks
      * `woe` is recomputed for EVERY bin, the missing bin included, with woe_from_counts
        and the same `smoothing`
      * ValueError for a direction other than "increasing" or "decreasing", or if a value bin
        is empty: its rate is undefined, so re-cut the edges rather than invent one

    Returns:
        a new WoeTable, with the same layout and at most as many value bins.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_monotonic() -> None:
    edges = np.array([-np.inf, 1.0, 2.0, 3.0, np.inf])
    ev = np.array([10, 26, 30, 12, 5])
    ne = np.array([90, 74, 70, 88, 5])
    t = WoeTable(edges, ev, ne, woe_from_counts(ev, ne))
    up = enforce_monotonic(t, "increasing")
    assert up.events.tolist() == [10, 68, 5] and up.non_events.tolist() == [90, 232, 5], (
        f"events {up.events.tolist()} — the docstring's example: pooling 0.30 with 0.12 gives "
        "0.21, now below the 0.26 before it, so the merged bin must be pooled again"
    )
    assert np.array_equal(up.edges, [-np.inf, 1.0, np.inf]), (
        f"edges {up.edges} — keep the outer edges and only the edges between pooled blocks"
    )
    assert np.allclose(up.woe, woe_from_counts(up.events, up.non_events)), (
        "recompute every WoE, the missing bin's included, from the pooled counts"
    )
    down = enforce_monotonic(t, "decreasing")
    rates = down.events[:-1] / (down.events[:-1] + down.non_events[:-1])
    assert np.all(np.diff(rates) <= 0), f"'decreasing' must never rise, got rates {rates}"
    assert down.events[-1] == 5 and down.non_events[-1] == 5, (
        "the missing bin is not on the axis: it is never pooled and keeps its counts"
    )
    print("exercise 5 looks right")

In [ ]:
_try("exercise 5", _check_monotonic)

Now the scorecard: five quantile bins per input, cut once on the development book, pooled
to the documented direction, weighed, and fitted with your own IRLS on the weights. With
this lesson's sign convention a higher weight means safer, so every coefficient should come
out negative; the cell checks that rather than assuming it.

In [ ]:
SCORECARD_SPEC = (("utilisation", "increasing"), ("income", "decreasing"),
                  ("delinquencies", "increasing"), ("months_on_book", "decreasing"))


def apply_woe(x: np.ndarray, table: WoeTable) -> np.ndarray:
    """Replace each value by its bin's weight of evidence; a NaN takes the missing bin's."""
    xa = np.asarray(x, dtype=float)
    n_value = table.edges.size - 1
    out = np.empty(xa.size)
    miss = np.isnan(xa)
    idx = np.clip(np.searchsorted(table.edges, xa[~miss], side="right") - 1, 0, n_value - 1)
    out[~miss] = table.woe[:n_value][idx]
    out[miss] = table.woe[-1]
    if np.isnan(out).any():
        raise ValueError("a value fell in a bin with no development evidence")
    return out


def scorecard_design(tables: Mapping[str, WoeTable],
                     inputs: Mapping[str, np.ndarray]) -> DesignMatrix:
    """The scorecard's design matrix: every input replaced by its bin's weight of evidence."""
    return design_matrix({f"woe_{name}": apply_woe(inputs[name], tables[name])
                          for name, _ in SCORECARD_SPEC})


def _show_scorecard() -> None:
    tables = {}
    for name, direction in SCORECARD_SPEC:
        x = DEV[name]
        raw = woe_table(x, DEV["y"], quantile_edges(x[~np.isnan(x)], N_WOE_BINS))
        tables[name] = enforce_monotonic(raw, direction)
        rates = raw.events / np.maximum(raw.events + raw.non_events, 1)
        print(f"{name:<15} {direction:<10} raw bins {raw.edges.size - 1}, pooled to "
              f"{tables[name].edges.size - 1}; raw event rates "
              + " ".join(f"{r:.3f}" for r in rates[:-1])
              + (f" | missing {rates[-1]:.3f}" if raw.events[-1] + raw.non_events[-1] else ""))
    design = scorecard_design(tables, DEV)
    fit = fit_logistic_irls(design, DEV["y"])
    print(f"\n{fit.finding}")
    for name, value in zip(fit.names, fit.coef):
        print(f"  {name:<20} {value:+.4f}")
    wrong = [n for n, v in zip(fit.names[1:], fit.coef[1:]) if v >= 0.0]
    print("every weight enters with the documented sign: " + ("yes" if not wrong else
          "NO — " + ", ".join(wrong) + " would reverse its own bins' direction"))
    _BUILT["scorecard"] = (tables, fit)


_try("the scorecard", _show_scorecard,
     needs=("exercise 1", "exercise 2", "exercise 4", "exercise 5"))

## 7. The leaderboard, and the question it does not ask

The 2026 interagency guidance on model risk names benchmarking to other models among the
assessments that can support a model's conceptual soundness. A benchmark is evidence only
if the comparison was fair. Score both challengers on the validation book and line all four
models up against the first-line extract, each on its own file, with module 1's rule: this
is what a leaderboard does.

In [ ]:
def _show_leaderboard() -> None:
    if not _built("irls", "scorecard"):
        return
    design, fit = _BUILT["irls"]
    irls_score = _sigmoid(design_matrix(challenger_columns(VAL, CHALLENGER_FEATURES)).X
                          @ fit.coef)
    tables, card = _BUILT["scorecard"]
    card_score = _sigmoid(scorecard_design(tables, VAL).X @ card.coef)
    _BUILT["frames"] = (VENDOR_FRAME, make_frame("irls-challenger", VAL, irls_score),
                        make_frame("scorecard-challenger", VAL, card_score))
    champ = {"auc": auc_by_ranks(CHAMPION_EXTRACT.y, CHAMPION_EXTRACT.score),
             "ece": expected_calibration_error(CHAMPION_EXTRACT.y, CHAMPION_EXTRACT.score)}
    print(f"{CHAMPION_EXTRACT.model:<34} AUC {champ['auc']:.4f}  ECE {champ['ece']:.4f}")
    for frame in _BUILT["frames"]:
        mine = {"auc": auc_by_ranks(frame.y, frame.score),
                "ece": expected_calibration_error(frame.y, frame.score)}
        verdict = challenger_decision(champ, mine, POLICY).verdict
        print(f"{frame.model:<34} AUC {mine['auc']:.4f}  ECE {mine['ece']:.4f}  "
              f"-> {verdict.upper()}")
    print("\nThe challenger you built looks promotable. Before anyone says so: was it given "
          "the\nsame records, the same period and the same missing values as the champion?")


_try("the leaderboard", _show_leaderboard,
     needs=("exercise 1", "exercise 2", "exercise 4", "exercise 5"))

## 8. Exercise 6 — `comparability_audit()`

Every committee asks it, and most comparisons answer it with a sentence. This one answers
with counts. Two frames are comparable when three things were held equal:

- **records** — the same accounts, each once, with the same outcome;
- **period** — the inputs of each shared record taken at the same date;
- **missing values** — the same values missing in the inputs each model was *given*.

The third needs care. What a model DOES with a missing value — an imputation, an indicator,
a bin of its own — is part of the model, and is exactly what the comparison is meant to
judge. What a pipeline did to the data BEFORE the model saw it is not: if one model was
handed a filled-in income and the other the gap, the comparison measures two pipelines.
Frames also arrive in different orders — the vendor sorted theirs by score — so records
are matched by id, never by position.

<details><summary>💡 Hint 1 — what to think about</summary>

Every check is about the records the two frames share, matched by id, so work out that
matching once. Then think about each check's trap: two sets of ids with the same size can
still differ, and an id can appear twice; two books can share a date range and still be
scored at different dates record by record; and a missing value is never equal to itself,
so comparing raw values calls every clean pair different. The evidence carries the counts.

</details>
<details><summary>💡 Hint 2 — the approach in words</summary>

Count duplicates in each frame, then take the ids only in the reference, only in the other,
and in both. Build, for the shared ids, the position of each in both frames; compare the
outcomes and the periods there, counting disagreements. For each input in the reference,
compare the two missingness masks on the shared records: count the records missing in
each, the records where only one side has a gap, and the distinct values the other side
holds in them. A check holds when its counts are all zero; write the counts into its
evidence either way.

</details>

In [ ]:
class Check(NamedTuple):
    """One thing that was, or was not, held equal — with the counts that prove it."""

    name: str       # "records" | "period" | "missing values"
    held: bool
    evidence: str


class Audit(NamedTuple):
    """Whether two frames can be compared at all, and why."""

    comparable: bool    # every check held
    checks: tuple       # (records, period, missing values), in that order


def comparability_audit(reference: EvalFrame, other: EvalFrame) -> Audit:
    """Was `other` given the same records, period and missing values as `reference`?

    The three checks, in this order, each graded:
      1. "records" — the same set of record ids, each exactly once in each frame, and the
         same outcome `y` for every record. Match records BY ID: frames may be in any order.
      2. "period" — on the records both frames hold, the same period for every record.
         Record by record: two books with the same date RANGE can still differ record by
         record.
      3. "missing values" — on the records both frames hold, and for every input named in
         the reference: missing (NaN) in exactly the same records. NaN == NaN is False, so
         compare the missingness masks, not the raw arrays. An input the other frame does
         not carry fails the check.
    If the frames share no records, checks 2 and 3 cannot be evaluated and do not hold.

    Evidence, graded for its numbers. Each check's evidence states how many records it
    compared and, when it fails, how many records fail it:
      * records — ids in common, ids only in the reference, ids only in the other,
        duplicated ids, and outcomes that disagree;
      * period — how many shared records carry a different period;
      * missing values — per input, how many shared records are missing in each frame, how
        many records differ, and for those, the distinct values the other side holds there:
        a column of one repeated value where the other frame has a gap is an imputation.

    Worked example:
        reference ids [1, 2, 3]; other ids [3, 2, 1], same periods, outcomes and NaNs
            -> comparable: order is not a difference
        other ids [1, 2]
            -> "records" does not hold: 1 id only in the reference

    Returns:
        Audit(comparable, (records check, period check, missing-values check)).
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _toy_frame(ids, period, income, y, score, model="toy") -> EvalFrame:
    return EvalFrame(model, np.asarray(ids), np.asarray(period),
                     {"income": np.asarray(income, dtype=float)}, np.asarray(y),
                     np.asarray(score, dtype=float))


def _check_audit() -> None:
    ref = _toy_frame([1, 2, 3, 4], ["2025-01"] * 4, [10.0, np.nan, 30.0, np.nan],
                     [0, 1, 0, 1], [0.1, 0.8, 0.2, 0.7])
    same = _toy_frame([4, 3, 2, 1], ["2025-01"] * 4, [np.nan, 30.0, np.nan, 10.0],
                      [1, 0, 1, 0], [0.6, 0.3, 0.9, 0.2])
    a = comparability_audit(ref, same)
    assert isinstance(a, Audit) and [c.name for c in a.checks] == ["records", "period",
                                                                    "missing values"], (
        "return Audit(comparable, checks) with the three checks in the order "
        "records, period, missing values"
    )
    assert a.comparable, (
        "the same four records in reverse order, with NaN in the same records, are comparable: "
        + " | ".join(f"{c.name}: {c.evidence}" for c in a.checks if not c.held)
        + " — match by id, not by position, and compare missingness masks, since NaN != NaN"
    )
    fewer = comparability_audit(ref, _toy_frame([1, 2, 3], ["2025-01"] * 3,
                                                [10.0, np.nan, 30.0], [0, 1, 0], [0.1, 0.8, 0.2]))
    assert not fewer.checks[0].held and fewer.checks[1].held and fewer.checks[2].held, (
        "one record dropped: only 'records' fails; period and missing values are judged on "
        "the records the frames share"
    )
    filled = comparability_audit(ref, _toy_frame([1, 2, 3, 4], ["2025-01"] * 4,
                                                 [10.0, 20.0, 30.0, 20.0], [0, 1, 0, 1],
                                                 [0.1, 0.8, 0.2, 0.7]))
    m = filled.checks[2]
    assert not m.held and "2" in m.evidence, (
        f"two missing incomes were filled: 'missing values' must fail and say how many — "
        f"got held={m.held}, evidence {m.evidence!r}"
    )
    later = comparability_audit(ref, _toy_frame([1, 2, 3, 4], ["2025-01", "2025-01", "2025-01",
                                                               "2025-04"],
                                                [10.0, np.nan, 30.0, np.nan], [0, 1, 0, 1],
                                                [0.1, 0.8, 0.2, 0.7]))
    assert not later.checks[1].held, "one record scored in a later month: 'period' must fail"
    print("exercise 6 looks right")

In [ ]:
_try("exercise 6", _check_audit)

In [ ]:
def _show_audit() -> None:
    audit = comparability_audit(CHAMPION_EXTRACT, VENDOR_FRAME)
    print(f"{VENDOR_FRAME.model} against {CHAMPION_EXTRACT.model}: "
          f"{'COMPARABLE' if audit.comparable else 'NOT COMPARABLE'}")
    for check in audit.checks:
        print(f"  {'HELD    ' if check.held else 'NOT HELD'} {check.name}: {check.evidence}")


_try("the audit", _show_audit, needs=("exercise 6",))

The vendor was not the problem. The first line's pipeline filled every missing income
before the champion saw it, so the champion's own missing-income term never fired — and
every challenger handed the raw gap was compared against a champion scored on different
data. The leaderboard in section 7 measured the pipeline, not the model.

## 9. Exercise 7 — `challenger_comparison()`

The comparison runs the audit first and only then measures, with the two instruments this
programme has already built: module 4's paired bootstrap of the AUC difference, because the
models are scored on the same records, and module 1's decision rule, because the rule was
written before the numbers were seen. Promotion needs both: the policy's margin, and an
interval that can tell the gain from zero. A challenger that fails the audit gets no
numbers at all.

<details><summary>💡 Hint 1 — what to think about</summary>

The order of operations is the exercise: audit, then align, then measure, then decide. A
non-comparable challenger has no metrics to compute, not merely a verdict to override. The
paired bootstrap must see the challenger's scores in the champion's record order, or it
pairs one account's score with another's outcome. And a challenger's interval must not
depend on which challengers happened to be compared before it.

</details>
<details><summary>💡 Hint 2 — the approach in words</summary>

For each challenger in turn: audit it against the champion; if it is not comparable, emit a
row that says so with every measurement empty. Otherwise find, for each champion record,
where that id sits in the challenger's frame and reorder the challenger's scores to match.
Measure AUC and ECE for both on the champion's labels, apply the decision rule, and run the
paired bootstrap with the challenger first and a generator seeded afresh for this challenger.
Reject if the rule rejects; promote only if it promotes and the interval's lower end is above
zero; hold otherwise.

</details>

In [ ]:
class ComparisonRow(NamedTuple):
    """One challenger against the champion: what was held equal, what was measured, the verdict."""

    model: str
    audit: Audit
    verdict: str                     # "promote" | "hold" | "reject" | "not comparable"
    champion_metrics: dict | None    # {"auc", "ece"} on the shared records; None if not comparable
    challenger_metrics: dict | None  # the same, for the challenger
    decision: Decision | None        # module 1's rule on those metrics
    paired: PairedDifference | None  # module 4's paired bootstrap, challenger minus champion


def challenger_comparison(champion: EvalFrame, challengers: Sequence[EvalFrame],
                          policy: Mapping[str, float] = POLICY, n_boot: int = N_BOOT,
                          alpha: float = ALPHA, seed: int = SEED) -> tuple:
    """Compare every challenger with the champion — or refuse to.

    For each challenger, in the order given:
      1. audit = comparability_audit(champion, challenger). If it is not comparable, the
         row's verdict is "not comparable" and its metrics, decision and paired difference
         are all None. Compute nothing: a number measured on non-comparable data is the
         number a committee will quote.
      2. otherwise put the challenger's scores into the CHAMPION'S record order, by id, and
         measure {"auc": auc_by_ranks(...), "ece": expected_calibration_error(...)} for both
         models on the champion's outcomes.
      3. decision = challenger_decision(champion_metrics, challenger_metrics, policy)
      4. paired = paired_bootstrap_difference(champion.y, challenger_scores, champion.score,
         n_boot, alpha, rng=np.random.default_rng(seed)) — challenger FIRST, so a positive
         difference favours it, and a generator seeded afresh for EVERY challenger, so no
         challenger's interval depends on which others were compared before it.
      5. verdict: "reject" if the decision rejects; "promote" only if the decision promotes
         AND paired.lo > 0, strictly — an interval that starts at exactly zero has not left
         it; otherwise "hold".

    Worked example: a challenger whose AUC gain clears the policy's margin but whose paired
    interval runs from -0.004 to +0.051 is a "hold" — the gain is not yet distinguishable
    from zero. The same challenger scored on a book taken three months later is "not
    comparable", with no metrics at all.

    Returns:
        tuple of ComparisonRow, one per challenger, in the order given.
    """
    # YOUR CODE HERE
    raise NotImplementedError


def _check_comparison() -> None:
    rng = np.random.default_rng(11)
    n = 600
    z = rng.normal(0.0, 1.0, n)
    y = (rng.random(n) < _sigmoid(-1.0 + 1.5 * z)).astype(np.int64)
    ids = np.arange(1, n + 1)
    per = np.array(["2025-01"] * n)
    inc = np.where(rng.random(n) < 0.1, np.nan, 1.0)
    champ = _toy_frame(ids, per, inc, y, _sigmoid(-1.0 + 0.6 * (z + rng.normal(0, 1.2, n))),
                       "toy champion")
    good_score = _sigmoid(-1.0 + 1.5 * z)
    good = _toy_frame(ids, per, inc, y, good_score, "good")
    late = _toy_frame(ids, np.array(["2025-04"] * n), inc, y, good_score, "good, scored later")
    rows = challenger_comparison(champ, [good, late], POLICY, n_boot=200)
    assert len(rows) == 2 and rows[0].verdict == "promote", (
        f"a well-calibrated challenger far better on the same records should promote, got "
        f"{rows[0].verdict!r} — is the paired difference challenger minus champion?"
    )
    assert rows[1].verdict == "not comparable" and rows[1].decision is None \
        and rows[1].paired is None and rows[1].challenger_metrics is None, (
            "the same scores taken three months later are NOT comparable: no metrics, no "
            f"decision, no interval — got verdict {rows[1].verdict!r}"
        )
    back = np.arange(n)[::-1]
    shuffled = EvalFrame("good", ids[back], per[back], {"income": inc[back]}, y[back],
                         good_score[back])
    again = challenger_comparison(champ, [shuffled], POLICY, n_boot=200)[0]
    assert again.paired is not None and abs(again.paired.diff - rows[0].paired.diff) < 1e-12, (
        "the same challenger in reverse record order must give the same result — put its scores "
        "into the champion's record order BY ID before pairing"
    )
    print("exercise 7 looks right")

In [ ]:
_try("exercise 7", _check_comparison)

In [ ]:
def render_comparison(champion: EvalFrame, rows: Sequence[ComparisonRow],
                      alpha: float = ALPHA) -> str:
    """The committee's page, generated from the comparison — pass the `alpha` it ran with."""
    lines = [f"# Challenger comparison — against {champion.model}", "", DATA_NOTE, ""]
    for row in rows:
        lines += [f"## {row.model}", "", "What was held equal, with the proof:"]
        lines += [f"- {'HELD' if c.held else 'NOT HELD'} · {c.name}: {c.evidence}"
                  for c in row.audit.checks]
        if row.verdict == "not comparable":
            lines += ["", "No metric is reported for this challenger. Re-run the comparison on "
                      "data prepared the same way for both models.", "Verdict: NOT COMPARABLE", ""]
            continue
        p = row.paired
        lines += ["", f"On the {len(champion.record_id)} shared records: champion AUC "
                  f"{row.champion_metrics['auc']:.4f}, ECE {row.champion_metrics['ece']:.4f}; "
                  f"challenger AUC {row.challenger_metrics['auc']:.4f}, "
                  f"ECE {row.challenger_metrics['ece']:.4f}.",
                  f"Paired AUC difference (challenger - champion): {p.diff:+.4f}, "
                  f"{100 * (1 - alpha):.0f}% interval [{p.lo:+.4f}, {p.hi:+.4f}] "
                  f"from {p.n_boot} paired replicates."]
        lines += [f"- {reason}" for reason in row.decision.reasons]
        lines += [f"Verdict: {row.verdict.upper()}", ""]
    return "\n".join(lines)


def _show_refusal() -> None:
    if not _built("frames"):
        return
    rows = challenger_comparison(CHAMPION_EXTRACT, _BUILT["frames"])
    for row in rows:
        failed = [c.name for c in row.audit.checks if not c.held]
        print(f"{row.model:<22} {row.verdict.upper():<15} not held: {', '.join(failed) or '-'}")


_try("the comparison, as delivered", _show_refusal,
     needs=("exercise 1", "exercise 2", "exercise 4", "exercise 5", "exercise 6", "exercise 7"))

Your code refused. The fix is not to relax the audit; it is to score the champion on the
same prepared data as everyone else. The champion's documented rule is in the model
inventory, so re-score it from the raw validation inputs — gaps and all — and run the
comparison again. This is the page that goes to the committee.

In [ ]:
def _show_comparison() -> None:
    if not _built("frames"):
        return
    rescored = make_frame("champion-v3 (re-scored from the inventory)", VAL, champion_model(VAL))
    rows = challenger_comparison(rescored, _BUILT["frames"])
    print(render_comparison(rescored, rows))
    naive = auc_by_ranks(CHAMPION_EXTRACT.y, CHAMPION_EXTRACT.score)
    honest = auc_by_ranks(rescored.y, rescored.score)
    print(f"the champion's AUC was {naive:.4f} on the first-line extract and {honest:.4f} "
          "re-scored from its documented rule: the difference was the pipeline's.")


_try("the comparison", _show_comparison,
     needs=("exercise 1", "exercise 2", "exercise 4", "exercise 5", "exercise 6", "exercise 7"))

## 10. Common mistakes

- **Reporting a coefficient under separation.** It is not an estimate; it is how far the
  optimiser walked before something stopped it. Section 3 showed a deviance rule stopping
  the walk and calling the result converged.
- **Inverting `X'WX`.** Solve the system. An inverse is slower, less accurate, and hides a
  near-singular matrix behind numbers that look fine.
- **Stopping when the deviance stops moving.** Under separation the deviance settles while
  a coefficient grows without bound. Watch the coefficients.
- **Trusting a fitter because it agrees with itself.** Check the score equations, and check
  a case with a closed form.
- **Pooling bins once.** A merged bin can violate the direction with its other neighbour.
- **Imputing inside a WoE table.** A missing value gets its own bin, with its own evidence.
- **Comparing frames by position.** A file can arrive in any order — the vendor's arrived
  sorted by its own score. Match by id.
- **`np.array_equal` on data with gaps in it.** NaN is not equal to itself, so every clean
  comparison fails — and a check that always fails teaches its readers to ignore it.
- **A verdict on non-comparable data.** "Promote, subject to data differences" is a
  promotion. The only verdict is "not comparable", with no numbers attached.

The NaN one is worth seeing rather than believing.

In [ ]:
def _show_nan_trap() -> None:
    income = VAL["income"]
    print(f"the validation incomes, compared with themselves by np.array_equal: "
          f"{np.array_equal(income, income)}")
    print(f"  ... with equal_nan=True: {np.array_equal(income, income, equal_nan=True)}")
    print(f"  ... by missingness masks: {np.array_equal(np.isnan(income), np.isnan(income))}")
    print("\nThe first line reports that a book differs from itself. A comparability check "
          "built on it\nfails every clean comparison, including this one.")


_try("NaN is not equal to itself", _show_nan_trap)

## 11. Self-check

1. An IRLS fit uses its whole iteration budget. One coefficient has grown by about one
   unit every update; the deviance has not moved in the fourth decimal place for the last
   ten. The entry for the findings register is:
   - (a) converged; the deviance is stable, so the coefficient is the estimate
   - (b) separation: that variable predicts some records perfectly and has no finite
         estimate; find out why before refitting without it
   - (c) not converged; raise the iteration limit to 100 and report what comes out

2. Your hand-built fitter and the 2×2 closed form disagree on a binary predictor's slope in
   the third decimal place. The most likely cause is:
   - (a) the closed form is only an approximation for large samples
   - (b) the 2×2 table has an empty cell
   - (c) your fitter stopped before the maximum, or maximised the wrong function

3. The scorecard's four equal-sized income bins have event rates 0.24, 0.19, 0.21, 0.12
   from lowest to highest income, and the documented direction is decreasing. Pooling
   adjacent violators produces:
   - (a) 0.24, 0.20 (the middle two pooled), 0.12
   - (b) 0.24, 0.19, 0.12 (the violating bin dropped)
   - (c) nothing — a direction is a documentation matter, not a binning one

4. The challenger beats the champion by 0.031 AUC. The audit shows the champion's extract
   was scored with missing incomes filled and the challenger's was not. The committee
   paper should say:
   - (a) promote, noting a difference in data preparation
   - (b) not comparable; no metrics reported until both are scored on the same prepared
         data
   - (c) hold, because the difference in preparation adds uncertainty

5. Two frames hold the same account ids, and their periods span the same range of
   months. The period check can still fail because:
   - (a) one frame's inputs for some accounts were taken in a different month within the
         range
   - (b) the frames list the records in different orders
   - (c) it cannot fail; the same range means the same period

Answers are in this lesson's worked solution in the course repository.

In [ ]:
_MARKS = {"passed": "✅", "failed": "❌", "not started": "⏳"}


def _progress_board() -> None:
    """One line per exercise, from the latest run of its check, then the tally."""
    width = max(len(", ".join(funcs)) for funcs in _EXERCISES.values())
    print("progress board")
    for label, funcs in _EXERCISES.items():
        state = _STATUS.get(label, "not started")
        print(f"  {_MARKS[state]} {label:<12} {', '.join(funcs):<{width}}  {state}")
    done = sum(_STATUS.get(label) == "passed" for label in _EXERCISES)
    print(f"\n{done} of {len(_EXERCISES)} exercises complete")
    failing = [label for label in _EXERCISES if _STATUS.get(label) == "failed"]
    if failing:
        print("failing right now: " + ", ".join(failing) + ". Each one printed what went "
              "wrong in its own cell above, and every exercise has hints you can open.")
    elif done < len(_EXERCISES):
        print("work top to bottom: every exercise has hints you can open above its code.")

## What you built, and where it goes next

Seven exercises: a design matrix that refuses what it cannot hold, a logistic regression
that reports separation instead of a runaway number, two proofs that a fit is the maximum,
a weight-of-evidence scorecard that obeys its documented directions, a comparability audit
that answers the committee's first question with counts, and a comparison that refuses to
measure what cannot be compared. Module 8 of this programme asks the next question of the
same challengers: whether the explanation of a score can be regenerated, byte for byte.

In [ ]:
# Your progress board. Every check is re-run here, quietly, against your code as it stands
# now — each one already printed its feedback in its own cell above — so the board is
# current even if you edited an exercise and did not re-run its check.
if __name__ == "__main__":
    with contextlib.redirect_stdout(io.StringIO()):
        for _name, _check in (("exercise 1", _check_design_matrix),
                              ("exercise 2", _check_irls),
                              ("exercise 3", _check_verification),
                              ("exercise 4", _check_woe),
                              ("exercise 5", _check_monotonic),
                              ("exercise 6", _check_audit),
                              ("exercise 7", _check_comparison)):
            _try(_name, _check)
    _progress_board()
    print(f"\nnotebook wall time so far: {time.perf_counter() - _LESSON_T0:.1f}s")
    # A stub you have not reached yet is not a failure. A check that ran and came back wrong
    # is: in a script or under CI it ends this run non-zero, rather than letting a green exit
    # code paper over it. Inside a notebook kernel the board above has already said so, in a
    # line rather than a traceback at the foot of the page.
    if _FAILED_CHECKS and "ipykernel" not in sys.modules:
        raise SystemExit("checks failed: " + ", ".join(dict.fromkeys(_FAILED_CHECKS)))